# 01 · Validation report

Fits the dependency-light baseline and the Bayesian TGI-survival model on a
training split and validates both against held-out patients: calibration,
IPCW Brier, CRPS, interval coverage, and Kaplan-Meier / log-rank. Produces
the citable metrics table and figures.

> Runs on synthetic data. Point the load cell at real PDS data to produce
> the paper's validation results.

In [ ]:
%matplotlib inline
import pandas as pd, matplotlib.pyplot as plt
from vca.data_processing.synthetic import make_synthetic_nsclc
from vca.models.baseline import MarginalResamplingModel
from vca.models.tgi_survival import TGISurvivalModel
from vca.validation.pipeline import run_validation, compare_models

td = make_synthetic_nsclc(500, seed=0)
train, test = td.train_test_split(0.3, seed=0)
print('train', train.n_patients, 'test', test.n_patients)

## Headline comparison — does the Bayesian model beat the baseline?

In [ ]:
models = {
    'baseline_resampling': MarginalResamplingModel(min_donors=8),
    'bayesian_tgi_survival': TGISurvivalModel(num_warmup=300, num_samples=300, num_chains=1, seed=0),
}
headline = compare_models(models, train, test, n_draws=300, seed=0, out_dir='results', make_figures=True)
headline

Higher `mean_logrank_p` = simulated survival curves are **less** distinguishable from real (good). Lower Brier / calibration ECE / CRPS = better. The full per-endpoint table is written to `results/model_comparison.csv`.

In [ ]:
pd.read_csv('results/model_comparison.csv')

## Figures (Kaplan-Meier overlays, calibration, SLD predictive bands)

In [ ]:
from IPython.display import Image, display
import glob, os
for f in sorted(glob.glob('results/figures/bayesian_tgi_survival_*.png')):
    print(os.path.basename(f)); display(Image(f))

## Model diagnostics
Always check NUTS convergence (R-hat ≈ 1.0, adequate ESS) before trusting the Bayesian results.

In [ ]:
m = models['bayesian_tgi_survival']
m.diagnostics()  # R-hat / ESS per parameter